In [2]:
import os
os.environ["HF_HOME"] = "C:/huggingface_cache"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from datasets import load_dataset
import pandas as pd

print("Loading Polyvore...")
ds_poly = load_dataset("Marqo/polyvore", split="data")
df_poly = ds_poly.remove_columns(["image"]).to_pandas()

print(f"Total rows: {len(df_poly)}")
print(f"Columns: {df_poly.columns.tolist()}")
print(f"\nSample rows:")
print(df_poly.head(10))

# Extract outfit ID and position
df_poly["outfit_id"] = df_poly["item_ID"].str.rsplit("_", n=1).str[0]
df_poly["position"]  = df_poly["item_ID"].str.rsplit("_", n=1).str[1]

print(f"\nUnique outfits: {df_poly['outfit_id'].nunique()}")
print(f"Unique items: {df_poly['item_ID'].nunique()}")
print(f"\nTop categories:")
print(df_poly["category"].value_counts().head(20))

c:\Users\shriy\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Polyvore...
Total rows: 94096
Columns: ['category', 'text', 'item_ID']

Sample rows:
       category                                           text      item_ID
0   Day Dresses                    tibi knit long sleeve dress  100002074_1
1         Boots       michael kors leather over-the-knee boots  100002074_2
2      Handbags  givenchy leather medium antigona duffel black  100002074_3
3    Sunglasses      bottega veneta acetate leather sunglasses  100002074_4
4  Floral Decor                              pier imports stem  100002074_5
5         Coats                                   miranda coat  100002074_7
6       Blazers                            three pocket blazer  100010727_1
7  Skinny Jeans                           givenchy skinny jean  100010727_2
8       Watches      guess black silver-tone chronograph watch  100010727_4
9    Sunglasses           ray-ban original wayfarer sunglasses  100010727_5

Unique outfits: 21587
Unique items: 94096

Top categories:
category
Ea

In [3]:
# Define which Polyvore categories are tops and bottoms
TOP_CATEGORIES = [
    "Tops", "T-Shirts", "Blouses", "Sweaters", "Sweatshirts",
    "Tank Tops", "Men's Shirts", "Blazers", "Jackets", "Coats",
    "Vests", "Intimates"
]

BOTTOM_CATEGORIES = [
    "Skinny Jeans", "Boyfriend Jeans", "Shorts", "Skirts",
    "Pants", "Leggings", "Joggers", "Trousers", "Jeans",
    "Day Dresses", "Rompers"
]

df_poly["garment_type"] = None
df_poly.loc[df_poly["category"].isin(TOP_CATEGORIES), "garment_type"]    = "top"
df_poly.loc[df_poly["category"].isin(BOTTOM_CATEGORIES), "garment_type"] = "bottom"

# Keep only tops and bottoms
df_poly_filtered = df_poly[df_poly["garment_type"].notna()].copy()

print(f"After filtering: {len(df_poly_filtered)} items")
print(df_poly_filtered["garment_type"].value_counts())
print(f"\nUnique outfits with tops/bottoms: {df_poly_filtered['outfit_id'].nunique()}")

After filtering: 22662 items
garment_type
top       14543
bottom     8119
Name: count, dtype: int64

Unique outfits with tops/bottoms: 14988


In [4]:
import random
random.seed(42)

pairs = []
seen  = set()

# Group by outfit
outfit_groups = df_poly_filtered.groupby("outfit_id")

# Positive pairs — top + bottom from SAME outfit
for outfit_id, group in outfit_groups:
    tops    = group[group["garment_type"] == "top"]["item_ID"].tolist()
    bottoms = group[group["garment_type"] == "bottom"]["item_ID"].tolist()
    
    if len(tops) == 0 or len(bottoms) == 0:
        continue
    
    for top in tops:
        for bottom in bottoms:
            pair_id = (top, bottom)
            if pair_id in seen:
                continue
            seen.add(pair_id)
            pairs.append({
                "item_A": top,
                "item_B": bottom,
                "label":  1   # REAL compatible pair
            })

print(f"Positive pairs created: {len(pairs)}")

# Negative pairs — top from one outfit + bottom from DIFFERENT outfit
all_tops    = df_poly_filtered[df_poly_filtered["garment_type"] == "top"]["item_ID"].tolist()
all_bottoms = df_poly_filtered[df_poly_filtered["garment_type"] == "bottom"]["item_ID"].tolist()

# Build outfit lookup for fast checking
item_to_outfit = dict(zip(df_poly_filtered["item_ID"], df_poly_filtered["outfit_id"]))

neg_count    = 0
neg_attempts = 0
target_negs  = len(pairs)  # equal positive and negative

while neg_count < target_negs:
    neg_attempts += 1
    if neg_attempts > target_negs * 20:
        print(f"Could only make {neg_count} negative pairs")
        break
    
    top    = random.choice(all_tops)
    bottom = random.choice(all_bottoms)
    
    # Make sure they're from DIFFERENT outfits
    if item_to_outfit[top] == item_to_outfit[bottom]:
        continue
    
    pair_id = (top, bottom)
    if pair_id in seen:
        continue
    
    seen.add(pair_id)
    pairs.append({
        "item_A": top,
        "item_B": bottom,
        "label":  0   # REAL incompatible pair
    })
    neg_count += 1

pairs_df_poly = pd.DataFrame(pairs)
print(f"\nTotal pairs: {len(pairs_df_poly)}")
print(f"Positive: {len(pairs_df_poly[pairs_df_poly['label'] == 1])}")
print(f"Negative: {len(pairs_df_poly[pairs_df_poly['label'] == 0])}")

Positive pairs created: 5969

Total pairs: 11938
Positive: 5969
Negative: 5969


In [5]:
# Check if images exist
ds_poly = load_dataset("Marqo/polyvore", split="data")
print(ds_poly.column_names)
print(ds_poly[0])

['image', 'category', 'text', 'item_ID']
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=274x400 at 0x20F16A7EBA0>, 'category': 'Day Dresses', 'text': 'tibi knit long sleeve dress', 'item_ID': '100002074_1'}


In [8]:
!pip install open-clip-torch --break-system-packages

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   --------------------------- ------------ 1.0/1.5 MB 6.8 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 5.7 MB/s  0:00:00
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   -------------------- ------------------- 1.3/2.6 MB 6.5 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 6.4 MB/s  0:00:00

   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ---------------------------------------- 0/2 [timm]
   ----------

In [6]:
import open_clip
import torch
from PIL import Image
import io
import numpy as np

# Load FashionCLIP
print("Loading FashionCLIP...")
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    'hf-hub:Marqo/marqo-fashionSigLIP'
)
clip_model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clip_model = clip_model.to(device)
print(f"FashionCLIP loaded on {device}")

KeyboardInterrupt: 

In [ ]:
# Get unique item IDs needed
unique_ids_needed = set(
    pairs_df_poly["item_A"].tolist() + 
    pairs_df_poly["item_B"].tolist()
)
print(f"Unique items needed: {len(unique_ids_needed)}")

# Filter dataset to only needed items
ds_poly_filtered = ds_poly.filter(
    lambda row: row["item_ID"] in unique_ids_needed
)
print(f"Items to extract: {len(ds_poly_filtered)}")

# Extract features
feature_dict_poly = {}
batch_size = 32
failed = 0

for start in range(0, len(ds_poly_filtered), batch_size):
    batch        = ds_poly_filtered[start : start + batch_size]
    batch_tensors = []
    batch_ids     = []
    
    for i in range(len(batch["item_ID"])):
        try:
            img    = batch["image"][i]
            if img.mode != "RGB":
                img = img.convert("RGB")
            tensor = preprocess(img)
            batch_tensors.append(tensor)
            batch_ids.append(batch["item_ID"][i])
        except:
            failed += 1
            continue
    
    if len(batch_tensors) == 0:
        continue
    
    batch_tensor = torch.stack(batch_tensors).to(device)
    
    with torch.no_grad():
        features = clip_model.encode_image(batch_tensor)
        features = features / features.norm(dim=-1, keepdim=True)  # normalize
    
    for i, item_id in enumerate(batch_ids):
        feature_dict_poly[item_id] = features[i].cpu().numpy()
    
    if (start // batch_size) % 50 == 0:
        print(f"Extracted {len(feature_dict_poly)}/{len(ds_poly_filtered)} | Failed: {failed}")

print(f"Done! Extracted: {len(feature_dict_poly)} | Failed: {failed}")

Unique items needed: 15264


Filter: 100%|██████████| 94096/94096 [01:33<00:00, 1010.81 examples/s]


Items to extract: 15264
Extracted 32/15264 | Failed: 0
Extracted 1632/15264 | Failed: 0
Extracted 3232/15264 | Failed: 0
Extracted 4832/15264 | Failed: 0
Extracted 6432/15264 | Failed: 0
Extracted 8032/15264 | Failed: 0
Extracted 9632/15264 | Failed: 0
Extracted 11232/15264 | Failed: 0
Extracted 12832/15264 | Failed: 0
Extracted 14432/15264 | Failed: 0


FileNotFoundError: [Errno 2] No such file or directory: 'D:/feature_dict_polyvore.npy'

Saved!


In [8]:
feature_dict_poly = np.load("C:/Users/shriy/Documents/feature_dict_polyvore.npy", allow_pickle=True).item()
print("Total extracted features:", len(feature_dict_poly))

sample_id = list(feature_dict_poly.keys())[0]
sample_vec = feature_dict_poly[sample_id]

print("Sample ID:", sample_id)
print("Vector shape:", sample_vec.shape)
print("Vector dtype:", sample_vec.dtype)

Total extracted features: 15264
Sample ID: 100002074_1
Vector shape: (768,)
Vector dtype: float32


In [ ]:
import numpy as np
np.save("C:/Users/shriy/Documents/feature_dict_polyvore.npy", feature_dict_poly)
print("Saved!")

In [10]:
pairs_df_poly["item_A"] = pairs_df_poly["item_A"].astype(str)
pairs_df_poly["item_B"] = pairs_df_poly["item_B"].astype(str)

pairs_df_poly["vec_A"] = pairs_df_poly["item_A"].map(feature_dict_poly)
pairs_df_poly["vec_B"] = pairs_df_poly["item_B"].map(feature_dict_poly)

print("Before drop:", len(pairs_df_poly))
print("Missing vec_A:", pairs_df_poly["vec_A"].isna().sum())
print("Missing vec_B:", pairs_df_poly["vec_B"].isna().sum())

pairs_df_poly = pairs_df_poly.dropna(subset=["vec_A", "vec_B"]).copy()

print("After drop:", len(pairs_df_poly))

Before drop: 11938
Missing vec_A: 0
Missing vec_B: 0
After drop: 11938


In [11]:
#train split and test split
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    pairs_df_poly,
    test_size=0.2,
    random_state=42,
    stratify=pairs_df_poly["label"]
)
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

Train: 9550, Val: 2388


In [12]:
#dataset and dataloader
from torch.utils.data import Dataset, DataLoader
import torch

class OutfitDataset(Dataset):
    def __init__(self, df):
        self.pairs = df.dropna(subset=["vec_A", "vec_B"]).reset_index(drop=True)
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        row = self.pairs.iloc[idx]
        vec_A = torch.tensor(row["vec_A"], dtype=torch.float32)
        vec_B = torch.tensor(row["vec_B"], dtype=torch.float32)
        label = torch.tensor(row["label"], dtype=torch.float32)
        return vec_A, vec_B, label

train_dataset = OutfitDataset(train_df)
val_dataset   = OutfitDataset(val_df)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Train batches: 299, Val batches: 75


In [21]:
class SiameseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )
    
    def forward(self, vec_A, vec_B):
        emb_A = self.embedding(vec_A)
        emb_B = self.embedding(vec_B)
        return emb_A, emb_B

model     = SiameseNetwork()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print("Model ready!")

Model ready!


In [22]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb_A, emb_B, label):
        # distance between two embeddings
        dist = torch.nn.functional.pairwise_distance(emb_A, emb_B)
        
        # positive pairs → distance should be small
        # negative pairs → distance should be > margin
        loss = (
            label * dist.pow(2) +
            (1 - label) * torch.clamp(self.margin - dist, min=0).pow(2)
        )
        return loss.mean()

criterion = ContrastiveLoss(margin=1.0)
print("Contrastive loss ready!")

Contrastive loss ready!


In [23]:
num_epochs       = 30
best_val_loss    = float('inf')
patience         = 5
patience_counter = 0

for epoch in range(num_epochs):

    # Training
    model.train()
    train_loss = 0
    for vec_A, vec_B, label in train_loader:
        emb_A, emb_B = model(vec_A, vec_B)
        loss = criterion(emb_A, emb_B, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    correct  = 0
    total    = 0

    with torch.no_grad():
        for vec_A, vec_B, label in val_loader:
            emb_A, emb_B = model(vec_A, vec_B)
            loss = criterion(emb_A, emb_B, label)
            val_loss += loss.item()

            # accuracy: small distance = good outfit
            dist      = torch.nn.functional.pairwise_distance(emb_A, emb_B)
            predicted = (dist < 0.5).float()  # close = compatible
            correct  += (predicted == label).sum().item()
            total    += label.size(0)

    avg_train = train_loss / len(train_loader)
    avg_val   = val_loss   / len(val_loader)
    accuracy  = correct / total * 100

    print(f"Epoch {epoch+1}/{num_epochs} → Train: {avg_train:.4f} | Val: {avg_val:.4f} | Acc: {accuracy:.1f}%")

    if avg_val < best_val_loss:
        best_val_loss    = avg_val
        patience_counter = 0
        torch.save(model.state_dict(), "C:/Users/shriy/Documents/best_model_contrastive.pth")
        print(f"  ✓ Best model saved!")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}!")
            break

print(f"\nBest val loss: {best_val_loss:.4f}")

Epoch 1/30 → Train: 12.4801 | Val: 2.5451 | Acc: 50.0%
  ✓ Best model saved!
Epoch 2/30 → Train: 1.3113 | Val: 0.3960 | Acc: 52.6%
  ✓ Best model saved!
Epoch 3/30 → Train: 0.4376 | Val: 0.2476 | Acc: 56.8%
  ✓ Best model saved!
Epoch 4/30 → Train: 0.3117 | Val: 0.2970 | Acc: 50.8%
  No improvement (1/5)
Epoch 5/30 → Train: 0.2873 | Val: 0.2747 | Acc: 53.2%
  No improvement (2/5)
Epoch 6/30 → Train: 0.2607 | Val: 0.2469 | Acc: 57.5%
  ✓ Best model saved!
Epoch 7/30 → Train: 0.2464 | Val: 0.2376 | Acc: 60.1%
  ✓ Best model saved!
Epoch 8/30 → Train: 0.2323 | Val: 0.2370 | Acc: 60.4%
  ✓ Best model saved!
Epoch 9/30 → Train: 0.2262 | Val: 0.2393 | Acc: 60.8%
  No improvement (1/5)
Epoch 10/30 → Train: 0.2181 | Val: 0.2373 | Acc: 60.8%
  No improvement (2/5)
Epoch 11/30 → Train: 0.2100 | Val: 0.2396 | Acc: 61.4%
  No improvement (3/5)
Epoch 12/30 → Train: 0.2019 | Val: 0.2354 | Acc: 61.2%
  ✓ Best model saved!
Epoch 13/30 → Train: 0.1975 | Val: 0.2414 | Acc: 61.8%
  No improvement (1/5)
E